# 01 Data Audit

Objetivo: validar qualidade, distribuição de classes, comprimento, turnos, duplicatas e sinais de leakage antes de qualquer fine tuning.

In [ ]:
import pandas as pd, json
from pathlib import Path

df = pd.read_csv("../data/raw/train.csv")
df.shape, df.columns.tolist()

In [ ]:
labels = ["winner_model_a", "winner_model_b", "winner_tie"]
pd.DataFrame({"count": df[labels].sum(), "share": df[labels].mean()})

In [ ]:
turns = df.prompt.map(lambda x: len(json.loads(x)))
turns.describe(percentiles=[.5,.9,.95,.99])

In [ ]:
audit = {
    "nulls": df.isna().sum().sort_values(ascending=False),
    "duplicate_ids": df.id.duplicated().sum(),
    "duplicate_pairs": df.duplicated(subset=["prompt","response_a","response_b"]).sum(),
    "unique_models": len(set(df.model_a).union(df.model_b)),
}
audit

Senior check: um split aleatório por linha pode vazar prompts repetidos. O pipeline usa hash do prompt normalizado como group id e separa grupos inteiros.